In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

In [ ]:
class Critic(nn.Module):
  def __init__(self,imgc,feat):
    super().__init__()
    self.crit=nn.Sequential(
        nn.Conv2d(imgc,feat,kernel_size=4,stride=2,padding=1),
        nn.LeakyReLU(0.2),
        self._block(feat,feat*2,4,2,1),
        self._block(feat*2,feat*4,4,2,1),
        self._block(feat*4,feat*8,4,2,1),
        nn.Conv2d(feat*8,1,kernel_size=4,stride=2,padding=0)
    )
  def _block(self,inn,out,kernel,stride,pad):
    return nn.Sequential(
        nn.Conv2d(inn,out,kernel,stride,pad,bias=False),
        nn.InstanceNorm2d(out,affine=True),
        nn.LeakyReLU(0.2)
    )
  def forward(self,x):
    return self.crit(x)

In [ ]:
class Generator(nn.Module):
  def __init__(self,noi,img,feat):
    super().__init__()
    self.gen=nn.Sequential(
        self._block(noi,feat*16,4,1,0),
        self._block(feat*16,feat*8,4,2,1),
        self._block(feat*8,feat*4,4,2,1),
        self._block(feat*4,feat*2,4,2,1),
        nn.ConvTranspose2d(feat*2,img,kernel_size=4,stride=2,padding=1),
        nn.Tanh()
    )
  def _block(self,inn,out,kern,stri,pad):
    return nn.Sequential(
        nn.ConvTranspose2d(inn,out,kern,stri,pad,bias=False),
        nn.BatchNorm2d(out),
        nn.ReLU()
    )
  def forward(self,x):
    return self.gen(x)

In [ ]:
def initalise(model):
  for m in model.modules():
    if isinstance(m,(nn.Conv2d,nn.ConvTranspose2d,nn.BatchNorm2d)):
      nn.init.normal_(m.weight.data,0.0,0.02)

In [ ]:
lr=5e-5
batch=64
imgsi=64
chnl=1
dim=128
featc=64
featg=64
clip=0.01

In [ ]:
transforms=transforms.Compose(
    [
        transforms.Resize(imgsi),
        transforms.ToTensor(),
        transforms.Normalize([0.5 for _ in range(chnl)],[0.5 for _ in range(chnl)])
    ]
)

In [ ]:
data = datasets.MNIST(root="dataset/", transform=transforms, download=True)

In [ ]:
loader=DataLoader(data,batch_size=batch,shuffle=True)

In [ ]:
gen=Generator(dim,chnl,featg)
critic=Critic(chnl,featc)

In [ ]:
initalise(gen)
initalise(critic)

In [ ]:
opt_gen = torch.optim.RMSprop(gen.parameters(), lr=lr)
opt_critic = torch.optim.RMSprop(critic.parameters(), lr=lr)

In [ ]:
fixed_noise = torch.randn(32, dim, 1, 1)
writer_real = SummaryWriter(f"logs/real")
writer_fake = SummaryWriter(f"logs/fake")
step = 0


In [ ]:
gen.train()
critic.train()

Critic(
  (crit): Sequential(
    (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.2)
    (2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.2)
    )
    (3): Sequential(
      (0): Conv2d(128, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): InstanceNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.2)
    )
    (4): Sequential(
      (0): Conv2d(256, 512, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): InstanceNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.2)
    )
    (5): Conv2d(512, 1, kernel_size=(4, 4), stride=(2, 2))
  )
)

In [ ]:
for epoch in range(5):
  gen.train()
  critic.train()
  for idx,(img,_) in enumerate(tqdm(loader)):
    cbs=img.shape[0]
    for _ in range(5):
      noi=torch.randn(cbs,dim,1,1)
      fake=gen(noi)
      creal=critic(img).reshape(-1)
      cfake=critic(fake).reshape(-1)
      lcri=-(torch.mean(creal)-torch.mean(cfake))
      critic.zero_grad()
      lcri.backward(retain_graph=True)
      opt_critic.step()
      for p in critic.parameters():
        p.data.clamp_(-clip,clip)
    gfake=critic(fake).reshape(-1)
    lgen=-torch.mean(gfake)
    gen.zero_grad()
    lgen.backward()
    opt_gen.step()
    if idx%100==0 and idx>0:
      gen.eval()
      critic.eval()
      print(f'Epoch  {epoch}|  Loss d  {lcri} | lsos g  {lgen}')

      with torch.no_grad():
        fake=gen(noi)
        rgrid=torchvision.utils.make_grid(img[:32],normalize=True)
        fgrid=torchvision.utils.make_grid(fake[:32],normalize=True)
        writer_real.add_image('real',rgrid,step)
        writer_fake.add_image('fake',fgrid,step)
      step+=1

 11%|█         | 100/938 [1:00:57<8:27:05, 36.31s/it]

Epoch  0|  Loss d  -1.3631778955459595 | lsos g  0.6575870513916016


 21%|██▏       | 200/938 [2:00:56<7:20:55, 35.85s/it]

Epoch  0|  Loss d  -1.4980273246765137 | lsos g  0.7248107194900513


 32%|███▏      | 300/938 [3:01:02<6:21:26, 35.87s/it]

Epoch  0|  Loss d  -1.526557207107544 | lsos g  0.7349007725715637


 38%|███▊      | 355/938 [3:34:00<5:47:05, 35.72s/it]

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs